In [1]:
# to run GUI event loop
%autosave 180
%matplotlib qt
import numpy as np
import nidaqmx
from nidaqmx import stream_writers
import matplotlib.pyplot as plt
import time
from tqdm import trange
from pypylon import pylon
from imageio import get_writer
import cv2


Autosaving every 180 seconds


In [39]:
############################################
############################################
############################################

#
try:
    grab.Release()
    camera.Close()
except:
    pass
#
tl_factory = pylon.TlFactory.GetInstance()
devices = tl_factory.EnumerateDevices()
for device in devices:
    print(device.GetFriendlyName())

tl_factory = pylon.TlFactory.GetInstance()
camera = pylon.InstantCamera()
camera.Attach(tl_factory.CreateDevice(devices[1]))
print("DeviceClass: ", camera.GetDeviceInfo().GetDeviceClass())
print("DeviceFactory: ", camera.GetDeviceInfo().GetDeviceFactory())
print("ModelName: ", camera.GetDeviceInfo().GetModelName())

# 
camera.Open()
camera.ExposureTime.SetValue(20000)  # exposure time in microseconds

camera.TriggerSource.SetValue("Line4")
camera.TriggerMode.SetValue("On")
camera.TriggerActivation.SetValue('RisingEdge')
camera.TriggerSelector.SetValue('FrameStart')

#camera.AcquisitionFrameRateEnable.SetValue(True);
#camera.AcquisitionFrameRate.SetValue(30.0);
    
# 
camera.StartGrabbing(pylon.GrabStrategy_LatestImageOnly)

# 
times = []
fps = 30

#
#f = open(r'D:\bmi\video_tests\video.bin','wb')

# #
# writer = get_writer(
#        r'D:\bmi\video_tests\video.mkv',  # mkv players often support H.264
#         fps=30,  # FPS is in units Hz; should be real-time.
#         codec='libx264',  # When used properly, this is basically "PNG for video" (i.e. lossless)
#         quality=None,  # disables variable compression
#         ffmpeg_params=[  # compatibility with older library versions
#             '-preset',   # set to fast, faster, veryfast, superfast, ultrafast
#             'fast',      # for higher speed but worse compression
#             '-crf',      # quality; set to 0 for lossless, but keep in mind
#             '24'         # that the camera probably adds static anyway
#         ]
# )


# initialize water image
height = 1200
width = 1824

# initialize video writer
fourcc = cv2.VideoWriter_fourcc('M','J','P','G')
fps = 30
video_filename = r'D:\bmi\video_tests\video.avi'
video_out = cv2.VideoWriter(video_filename, 
                            fourcc, 
                            fps, 
                            (width, height))

#
n_frames = 200
for k in trange(n_frames):
    try:
        grab = camera.RetrieveResult(10000, 
                                 pylon.TimeoutHandling_ThrowException)
        #writer.append_data()
        frame = grab.GetArray()
        
        #
        gray = cv2.normalize(frame, None, 255, 0, 
                             norm_type=cv2.NORM_MINMAX, 
                             dtype=cv2.CV_8U)
        gray_3c = cv2.merge([gray, gray, gray])
        video_out.write(gray_3c)
        
        #
        times.append([k ,time.time()])
    except:
        print ("TIME OUT")
        break

try:
    # close out the video writer
    video_out.release()
except:
    print ("WARNING VIDEO WRITER DIDN'T CLOSE")
try:
    camera.Close()
except:
    print ("WARNING CAMERA didn't close")

f.close()
plt.close()
np.save(r'D:\bmi\video_tests\video_times.npy', times)

print ("Done")


Basler acA1920-155um (22333978)
Basler acA1920-150um (40038869)
DeviceClass:  BaslerUsb
DeviceFactory:  USB/BaslerUsb 6.3.0.18933
ModelName:  acA1920-150um


100%|█████████████████████████████████████████████████████████████████████████████████████████████████████| 200/200 [00:10<00:00, 18.65it/s]

Done


In [ ]:
import pypylon.pylon as pylon
from imageio import get_writer

#fps = 5  # Hz
#time_to_record = 60  # seconds
#images_to_grab = fps * time_to_record
images_to_grab = 10000

tlf = pylon.TlFactory.GetInstance()
devices = tlf.EnumerateDevices()

cam = pylon.InstantCamera(tlf.CreateDevice(devices[0]))
cam.Open()
print("Using device ", cam.GetDeviceInfo().GetModelName())
cam.AcquisitionFrameRate.SetValue(fps)

writer = get_writer(
       'output-filename.mkv',  # mkv players often support H.264
        fps=fps,  # FPS is in units Hz; should be real-time.
        codec='libx264',  # When used properly, this is basically "PNG for video" (i.e. lossless)
        quality=None,  # disables variable compression
        ffmpeg_params=[  # compatibility with older library versions
            '-preset',   # set to fast, faster, veryfast, superfast, ultrafast
            'fast',      # for higher speed but worse compression
            '-crf',      # quality; set to 0 for lossless, but keep in mind
            '24'         # that the camera probably adds static anyway
        ]
)

print(f"Recording {time_to_record} second video at {fps} fps")
cam.StartGrabbingMax(images_to_grab, pylon.GrabStrategy_OneByOne)
while cam.IsGrabbing():
    with cam.RetrieveResult(1000, pylon.TimeoutHandling_ThrowException) as res:
        if res.GrabSucceeded():
            img = res.Array
            writer.append_data(img)
            print(res.BlockID, end='\r')
            res.Release()
        else:
            print("Grab failed")
            # raise RuntimeError("Grab failed")

print("Saving...", end=' ')
cam.StopGrabbing()
cam.Close()
print("Done")

In [3]:
import cv2

cap = cv2.VideoCapture(0)

# Check if the webcam is opened correctly
if not cap.isOpened():
    raise IOError("Cannot open webcam")

#
while True:
    ret, frame = cap.read()
    frame = cv2.resize(frame, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    cv2.imshow('Input', frame)

    c = cv2.waitKey(1)
    if c == 27:
        break

#
cap.release()
cv2.destroyAllWindows()

OSError: Cannot open webcam

In [7]:
############################################
############################################
############################################
#  Initialize nidaqmx Task
water_Task = nidaqmx.Task()

# 
water_Task.ao_channels.add_ao_voltage_chan('Dev3/ao1')
water_Task.timing.cfg_samp_clk_timing(rate = 1000, 
                                     #samps_per_chan=100,  # in continuos mode this is the buffer
                                     sample_mode=nidaqmx.constants.AcquisitionType.CONTINUOUS)
                                     #sample_mode=nidaqmx.constants.AcquisitionType.FINITE)

# 
water_Writer = nidaqmx.stream_writers.AnalogSingleChannelWriter(water_Task.out_stream, 
                                                                auto_start=True)

# duration of water release in 100,000Hz time base which is the default
# for 0.5 seconds, we set duration to 50,000 
duration = 200000  # time in 100,000Hz time base

#
voltage_level = 5   #

# release water 10 times
for k in range(5):
    
    # release water for specific duration
    start = time.time()
    for p in range(duration):
        water_Writer.write_one_sample(voltage_level)
    
    # reset the pulse to zero
    for q in range(100000):
        water_Writer.write_one_sample(0)

    #
    print ("pulse: ", k, " duration: ", time.time()-start, "sec")
#    
water_Task.stop()
water_Task.close()

print ("DONE")

pulse:  0  duration:  3.6813924312591553 sec
pulse:  1  duration:  3.675281047821045 sec
pulse:  2  duration:  3.70388126373291 sec
pulse:  3  duration:  3.7028093338012695 sec
pulse:  4  duration:  3.679234504699707 sec
DONE


In [59]:
data = np.load(r'D:\bmi\DON8460\22-07-07\databmi_results.npz')

In [60]:
print (data.files)
licks = data['lick_detector_abstime']
abstimes = data['abs_times']

plt.figure()
plt.plot(abstimes, licks)
plt.show()

['ttl_voltages', 'ttl_n_computed', 'ttl_n_detected', 'abs_times', 'ttl_times', 'rois_pixels', 'rois_traces_raw', 'rois_traces_smooth', 'reward_times', 'ensemble_activity', 'ensemble_diff_array', 'received_reward_lockout', 'max_reward_window', 'missed_reward_lockout', 'sampleRate_NI', 'ttl_pts', 'sampleRate_2P', 'image_width', 'image_length', 'max_n_seconds_session', 'n_frames', 'n_frames_to_be_acquired', 'rois_smooth_window', 'n_ttl_to_start_applying_DFF0_computation', 'n_frames_search_forward', 'drift_array', 'template', 'lick_detector_abstime', 'rotary_encoder1_abstime', 'rotary_encoder2_abstime']
